# 09 - Previsão operacional / hindcast da estação 413

Este notebook simula como a arquitetura v7 produz uma previsão em um horário específico.

A saída separa explicitamente:

1. **previsão de magnitude** (`ΔH` em 2 h);
2. **detector de regime** (decide se a magnitude vem do XGBoost ou do ensemble neural);
3. **sinais de severidade** `>=1/2/3 m` produzidos pela TCN.

Os sinais de severidade **não são alertas oficiais** e não usam cotas de atenção/emergência. O alvo continua sendo somente a estação 413.

In [ ]:
from pathlib import Path
import sys, json, warnings
import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore")
CWD=Path.cwd().resolve(); ROOT=CWD.parent if CWD.name.lower()=="notebooks" else CWD
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
PROCESSED=ROOT/"data"/"processed"; MODELS=ROOT/"models"; OUTPUTS=ROOT/"outputs"; PREDICTIONS=OUTPUTS/"predictions"
PREDICTIONS.mkdir(parents=True,exist_ok=True)

final_meta = MODELS / "final_model_metadata_v7.json"
if final_meta.exists():
    meta=json.loads(final_meta.read_text(encoding="utf-8")); MODEL_STAGE="final_v7"
else:
    arch_path=MODELS/"architecture_candidate_v7.json"
    if not arch_path.exists(): raise FileNotFoundError("Execute 07c primeiro; para modelos finais, execute também 08.")
    arch=json.loads(arch_path.read_text(encoding="utf-8")); MODEL_STAGE="development_v7"
    meta={
        "lookback_steps": int(json.loads((MODELS/"neural_registry_v6.json").read_text())["lookback_steps"]),
        "regime_probability_threshold": float(arch["point_forecast"]["regime_probability_threshold"]),
        "alert_thresholds": arch["severity_signals"]["thresholds_m"],
        "alert_persistence_steps": int(arch["severity_signals"]["persistence_steps"]),
    }
print("Modo:",MODEL_STAGE)

## Escolha o horário

Para um hindcast histórico, a célula final também mostra o que realmente aconteceu. Em operação real, naturalmente essa comparação ainda não existe.

In [ ]:
PREDICTION_TIME = pd.Timestamp("2024-02-14 01:20:00")
HORIZON_MIN = 120
TARGET_TIME = PREDICTION_TIME + pd.Timedelta(minutes=HORIZON_MIN)
print("Previsão emitida:",PREDICTION_TIME)
print("Horizonte:",TARGET_TIME)

In [ ]:
master=pd.read_parquet(PROCESSED/"master_base.parquet").sort_index()
features_radar=pd.read_parquet(PROCESSED/"features_causal_radar.parquet").sort_index()
radar=pd.read_parquet(PROCESSED/"radar_features_basic.parquet").sort_index()
if radar.index.has_duplicates: radar=radar.groupby(level=0).mean(numeric_only=True).sort_index()
if PREDICTION_TIME not in master.index: raise KeyError("Timestamp não encontrado.")

stage_now=float(master.loc[PREDICTION_TIME,"stage_413"])
print("Nível atual 413:",stage_now)

## 1. Previsão do XGBoost + radar

In [ ]:
from xgboost import XGBRegressor

xgb_file = MODELS/("final_xgboost_radar_120m_v7.json" if (MODELS/"final_xgboost_radar_120m_v7.json").exists() else "xgb_radar_dev_120m.json")
xgb=XGBRegressor(); xgb.load_model(xgb_file)
feature_cols=[c for c in features_radar.columns if c!="split" and not c.startswith("target_")]
row=features_radar.loc[[PREDICTION_TIME],feature_cols]
pred_xgb=float(xgb.predict(row)[0])
print("XGBoost + radar ΔH_2h:",f"{pred_xgb:+.3f} m")

## 2. TCN e GRU: sequência das últimas 6 h

Calculamos também alguns timestamps imediatamente anteriores para saber se os sinais neurais estão **persistentes**, em vez de reagir a um único pico de 10 minutos.

In [ ]:
import torch
from utils.temporal_models import (
    TemporalPreprocessor, TemporalPreprocessorState, build_temporal_frame,
    TCNMultiTask, GRUMultiTask, choose_device
)

raw,groups=build_temporal_frame(master,radar=radar)
lookback=int(meta["lookback_steps"])

pre_path = MODELS/"final_temporal_preprocessor_v7.json"
if pre_path.exists():
    state=TemporalPreprocessorState.from_json(pre_path)
else:
    state=TemporalPreprocessorState.from_json(MODELS/"temporal_preprocessor_v6.json")
pre=TemporalPreprocessor(groups,stage_ffill_limit=state.stage_ffill_limit); pre.state=state
scaled=pre.transform(raw); matrix=scaled.to_numpy("float32")

input_dim=matrix.shape[1]; device=choose_device()
tcn=TCNMultiTask(input_dim,channels=(64,64,96,96),kernel_size=3,dropout=.15)
gru=GRUMultiTask(input_dim,hidden_dim=96,num_layers=2,dropout=.15)

def load_state(model, final_name, dev_name):
    path=MODELS/final_name if (MODELS/final_name).exists() else MODELS/dev_name
    ckpt=torch.load(path,map_location=device,weights_only=False)
    model.load_state_dict(ckpt["state_dict"]); return model.to(device).eval(),path

tcn,tcn_path=load_state(tcn,"final_tcn_120m_v7.pt","tcn_multitask_dev_v6.pt")
gru,gru_path=load_state(gru,"final_gru_120m_v7.pt","gru_multitask_dev_v6.pt")
print("TCN:",tcn_path.name,"| GRU:",gru_path.name)

def predict_sequence(model, pos):
    start=pos-lookback+1
    if start<0: raise ValueError("Histórico insuficiente para a sequência.")
    x=torch.from_numpy(matrix[start:pos+1][None,:,:]).to(device)
    with torch.no_grad(): reg,logits=model(x)
    pred=float(pre.inverse_target(reg.detach().cpu().numpy())[0])
    probs=1/(1+np.exp(-np.clip(logits.detach().cpu().numpy()[0],-30,30)))
    probs[1]=min(probs[1],probs[0]); probs[2]=min(probs[2],probs[1])
    return pred,probs

pos=raw.index.get_loc(PREDICTION_TIME)
persistence=int(meta["alert_persistence_steps"])
positions=list(range(pos-persistence+1,pos+1))
neural_history=[]
for p in positions:
    pt=raw.index[p]; pred_t,probs_t=predict_sequence(tcn,p); pred_g,_=predict_sequence(gru,p)
    neural_history.append({"prediction_time":pt,"pred_tcn":pred_t,"pred_gru":pred_g,"p_ge_1m":probs_t[0],"p_ge_2m":probs_t[1],"p_ge_3m":probs_t[2]})
neural_history=pd.DataFrame(neural_history)
display(neural_history)

## 3. Detector de regime e previsão híbrida

O detector de regime usa **somente** `TCN p(ΔH>=1m)` no timestamp atual. Se o score ultrapassa o limiar congelado, a magnitude muda de XGBoost para `média(TCN,GRU)`.

Isso é um roteamento interno de modelos; não é uma classificação oficial de risco.

In [ ]:
current=neural_history.iloc[-1]
regime_threshold=float(meta["regime_probability_threshold"])
risk_regime=bool(current.p_ge_1m>=regime_threshold)
pred_neural_mean=float((current.pred_tcn+current.pred_gru)/2)
pred_hybrid=pred_neural_mean if risk_regime else pred_xgb
model_used="TCN+GRU" if risk_regime else "XGBoost+radar"

print("Score detector >=1m:",f"{current.p_ge_1m:.3f}","| limiar:",f"{regime_threshold:.3f}")
print("Regime:","subida forte" if risk_regime else "normal")
print("Modelo usado na magnitude:",model_used)
print("ΔH previsto em 2h:",f"{pred_hybrid:+.3f} m")
print("Nível previsto 413:",f"{stage_now+pred_hybrid:.3f} m")

## 4. Sinais de severidade com persistência

Um sinal só aparece como “persistente” se o threshold tiver sido superado nos últimos `N` timestamps consecutivos. Na v7, `N` é definido no 07c.

**Importante:** “sinal >=3 m” significa que o classificador identificou padrão compatível com uma subida de pelo menos 3 m nas próximas 2 h. Não significa cota de emergência.

In [ ]:
alert_thresholds={float(k):float(v) for k,v in meta["alert_thresholds"].items()}
cols={1.0:"p_ge_1m",2.0:"p_ge_2m",3.0:"p_ge_3m"}
signal_rows=[]
for sev,col in cols.items():
    values=neural_history[col].to_numpy()
    raw_active=bool(values[-1]>=alert_thresholds[sev])
    persistent_active=bool(len(values)>=persistence and np.all(values[-persistence:]>=alert_thresholds[sev]))
    signal_rows.append({
        "faixa_subida":f">={sev:g} m",
        "score_atual":float(values[-1]),
        "threshold":alert_thresholds[sev],
        "ativo_agora":raw_active,
        f"persistente_{persistence}x10min":persistent_active,
    })
signals=pd.DataFrame(signal_rows)
display(signals)
active_levels=[sev for sev,row in zip(cols.keys(),signal_rows) if row[f"persistente_{persistence}x10min"]]
max_signal=max(active_levels) if active_levels else 0.0
print("Maior faixa persistente ativa:", f">= {max_signal:g} m" if max_signal else "nenhuma")

## 5. Resumo humano e comparação retrospectiva

In [ ]:
result={
    "prediction_time":PREDICTION_TIME,
    "target_time":TARGET_TIME,
    "stage_now_413_m":stage_now,
    "pred_delta_2h_m":pred_hybrid,
    "pred_stage_413_2h_m":stage_now+pred_hybrid,
    "magnitude_model_used":model_used,
    "regime_score_ge_1m":float(current.p_ge_1m),
    "regime_threshold":regime_threshold,
    "risk_regime_internal":risk_regime,
    "persistent_severity_signal_m":max_signal,
    "official_stage_thresholds_used":False,
}
if TARGET_TIME in master.index and pd.notna(master.loc[TARGET_TIME,"stage_413"]):
    real_future=float(master.loc[TARGET_TIME,"stage_413"]); real_delta=real_future-stage_now
    result.update({"real_delta_2h_m":real_delta,"real_stage_413_2h_m":real_future,"error_cm":100*(pred_hybrid-real_delta)})
summary=pd.DataFrame([result]); display(summary.T)

stem=PREDICTION_TIME.strftime("%Y%m%d_%H%M")
summary.to_csv(PREDICTIONS/f"operational_hindcast_{stem}_v7.csv",index=False)
(PREDICTIONS/f"operational_hindcast_{stem}_v7.json").write_text(json.dumps({k:(str(v) if isinstance(v,pd.Timestamp) else v) for k,v in result.items()},indent=2,ensure_ascii=False),encoding="utf-8")

## Leitura correta da saída

- **Modelo usado na magnitude**: qual regressor forneceu o número final de metros.
- **Regime interno**: apenas escolhe XGBoost ou ensemble neural.
- **Sinal persistente de severidade**: classificação experimental da subida futura, não da cota atual do rio.
- Sem cotas oficiais, este notebook ainda **não pode afirmar** “atenção”, “emergência” ou “transbordamento”.